In [ ]:
import json
import sys
import tempfile
import zipfile
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

from pipeline.utils.helpers import load_run_config, load_json, parse_plan_district_rep_from_path
from pipeline.profile_generator import profile_class_for_mode
from votekit import RankProfile

CONFIG_NAME = "configs/basic.json"

config = load_run_config(PROJECT_DIR / CONFIG_NAME)
RUN_NAME = config["run_name"]
RUN_DIR = PROJECT_DIR / "outputs" / RUN_NAME
RUN_DIR

## Settings

In [ ]:
settings_rows = []
for settings_path in sorted(RUN_DIR.glob("settings/*/*.json")):
    plan, district, _ = parse_plan_district_rep_from_path(settings_path.name)
    row = {
        "district_count": int(settings_path.parent.name),
        "plan": plan,
        "district_id": district,
        "path": settings_path,
    }
    row.update(load_json(settings_path))
    settings_rows.append(row)

settings_df = pd.DataFrame(settings_rows)
settings_df

## Preference profiles

In [ ]:
with zipfile.ZipFile(RUN_DIR / "profiles.zip") as z:
    profile_members = z.namelist()

def load_profile(z, member):
    mode = member.split("/")[0]
    with tempfile.NamedTemporaryFile(mode="wb", suffix=".csv", delete=False) as tmp:
        tmp.write(z.read(member))
        tmp_path = Path(tmp.name)
    profile = profile_class_for_mode(mode).from_csv(tmp_path)
    tmp_path.unlink()
    return profile

ballot_rows = []
with zipfile.ZipFile(RUN_DIR / "profiles.zip") as z:
    for member in profile_members:
        mode = member.split("/")[0]
        plan, district, rep = parse_plan_district_rep_from_path(member)
        profile = load_profile(z, member)
        for ballot in profile.ballots:
            is_ranked = isinstance(profile, RankProfile)
            ranking = [
                "/".join(sorted(position)) if len(position) > 1 else next(iter(position))
                for position in ballot.ranking
            ] if is_ranked else None
            ballot_rows.append({
                "mode": mode,
                "plan": plan,
                "district_id": district,
                "rep": rep,
                "member": member,
                "ranking": ranking,
                "scores": None if is_ranked else dict(ballot.scores),
                "weight": ballot.weight,
            })

profiles_df = pd.DataFrame(ballot_rows)
profiles_df

## Election results

In [ ]:
results_rows = []
for results_path in sorted(RUN_DIR.glob("election_results/*/*.json")):
    mode = results_path.parent.name
    data = load_json(results_path)
    profile_files = data.get("profile_files", [])
    for idx, result in enumerate(data.get("election_results", [])):
        plan, district, rep = parse_plan_district_rep_from_path(profile_files[idx])
        for method, winners in result.items():
            results_rows.append({
                "mode": mode,
                "plan": plan,
                "district_id": district,
                "rep": rep,
                "election_method": method,
                "winners": winners,
                "n_winners": len(winners),
            })

results_df = pd.DataFrame(results_rows)
results_df